# MedGemma LoRA con dataset oficial

Este notebook entrena un adapter LoRA para descripcion usando las transcripciones expertas del dataset actualizado.

Comparacion principal:

```text
MedGemma base BERTScore en test
vs
MedGemma + LoRA BERTScore en test
```

Modo de ejecucion:

- En Colab usa `RUN_MODE="smoke"` por defecto para validar con pocos datos y pocos pasos.
- En Kabre usa `RUN_MODE="full"` por defecto para entrenar con todo el split de train.

Nota: esto usa LoRA puro con el modelo completo en GPU.


In [1]:
from pathlib import Path

REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"
REPO_NAME = "utils_medgemma"
MODEL_ID = "google/medgemma-1.5-4b-it"
SPLIT_FILE = "dataset/split_repetition_1.json"
LORA_OUTPUT_DIR = "checkpoints/official_medgemma_lora_description"


## 1. Setup: Colab o cluster


In [2]:
import os
import shutil
from pathlib import Path

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

ip = get_ipython()

if IS_COLAB:
    repo_dir = Path("/content") / REPO_NAME
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    ip.system(f'git clone "{REPO_URL}" "{repo_dir}"')
    ip.run_line_magic("cd", str(repo_dir))
    ip.system("pip install -q -r requirements.txt")
else:
    project_root = Path.cwd().resolve()
    if project_root.name == "notebooks":
        project_root = project_root.parent
    if not (project_root / "dataset").exists():
        raise RuntimeError("No encuentro dataset/. En Kabre corre el notebook desde la raiz del repo o ajusta project_root.")
    os.chdir(project_root)
    print(f"Ejecutando desde repo local: {project_root}")


Ejecutando desde repo local: C:\Users\Luco1421\Desktop\Pendientes\utils_medgemma


## 2. Imports, token y dataset

In [3]:
import json
import os
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import torch
from PIL import Image
os.environ.setdefault("MPLCONFIGDIR", str((Path(".cache") / "matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

from huggingface_hub import HfApi
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from bert_score import score as bertscore

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if IS_COLAB and not HF_TOKEN:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("Define HF_TOKEN en el entorno. En Kabre: export HF_TOKEN=...; en Colab: guardalo en Secrets.")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
RUN_MODE = os.environ.get("MEDGEMMA_RUN_MODE", "smoke" if IS_COLAB else "full").lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("MEDGEMMA_RUN_MODE debe ser 'smoke' o 'full'.")

TRAIN_LIMIT = int(os.environ.get("MEDGEMMA_TRAIN_LIMIT", "4" if RUN_MODE == "smoke" else "0")) or None
EVAL_LIMIT = int(os.environ.get("MEDGEMMA_EVAL_LIMIT", "2" if RUN_MODE == "smoke" else "0")) or None
MAX_STEPS = int(os.environ.get("MEDGEMMA_MAX_STEPS", "2" if RUN_MODE == "smoke" else "60"))
GRADIENT_ACCUMULATION_STEPS = int(os.environ.get("MEDGEMMA_GRAD_ACCUM", "1" if RUN_MODE == "smoke" else "4"))

api = HfApi()
hf_user = api.whoami(token=HF_TOKEN)
print("HF user:", hf_user.get("name") or hf_user.get("fullname"))
print("HF model:", api.model_info(MODEL_ID, token=HF_TOKEN).modelId)
assert torch.cuda.is_available(), "Se requiere GPU CUDA para correr MedGemma/LoRA."
print(torch.cuda.get_device_name(0))
print({
    "run_mode": RUN_MODE,
    "train_limit": TRAIN_LIMIT,
    "eval_limit": EVAL_LIMIT,
    "max_steps": MAX_STEPS,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
})


HF user: luco1421
HF model: google/medgemma-1.5-4b-it
NVIDIA GeForce RTX 3060
{'run_mode': 'smoke', 'train_limit': 1, 'eval_limit': 1, 'max_steps': 1, 'gradient_accumulation_steps': 1}


In [4]:
DATASET_ROOT = Path("dataset")
DESCRIPTION_PROMPT = "Describe the ophthalmological findings in this fundus image."

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)

def read_annotation(annotation_path):
    with open(DATASET_ROOT / annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data[0] if isinstance(data, list) else data

def make_row(split_name, item):
    ann = read_annotation(item["annotation"])
    conditions = ann.get("locs_data", {}).get("conditions", []) or []
    return {
        "split": split_name,
        "image": str(DATASET_ROOT / item["image"]),
        "annotation": str(DATASET_ROOT / item["annotation"]),
        "label": ann.get("label"),
        "conditions": conditions,
        "target_label": "glaucoma" if "glaucoma" in [c.lower() for c in conditions] else "non_glaucoma",
        "prompt": DESCRIPTION_PROMPT,
        "answer": ann.get("transcription", ""),
        "reference": ann.get("transcription", ""),
        "locs_data": ann.get("locs_data", {}),
    }

rows = []
for split_name in ["train", "validation", "test"]:
    rows.extend(make_row(split_name, item) for item in split_data[split_name])

train_rows_all = [r for r in rows if r["split"] == "train" and r["answer"]]
val_rows = [r for r in rows if r["split"] == "validation" and r["answer"]]
test_rows_all = [r for r in rows if r["split"] == "test" and r["answer"]]
train_rows = train_rows_all[:TRAIN_LIMIT] if TRAIN_LIMIT is not None else train_rows_all
test_rows = test_rows_all[:EVAL_LIMIT] if EVAL_LIMIT is not None else test_rows_all

print("splits:", Counter(r["split"] for r in rows))
print("labels:", Counter(r["label"] for r in rows))
print("targets:", Counter(r["target_label"] for r in rows))
print("train examples used:", len(train_rows), "/", len(train_rows_all))
print("test examples used:", len(test_rows), "/", len(test_rows_all))
train_rows[0]


splits: Counter({'train': 57, 'validation': 20, 'test': 19})
labels: Counter({'Pathological': 70, 'Normal': 13, 'Bad quality': 10, 'Needs dilation': 3})
targets: Counter({'glaucoma': 70, 'non_glaucoma': 26})
train examples used: 1 / 57
test examples used: 1 / 19


{'split': 'train',
 'image': 'dataset\\1494\\1494_left.jpg',
 'annotation': 'dataset\\1494\\1494_left.json',
 'label': 'Normal',
 'conditions': [],
 'target_label': 'non_glaucoma',
 'prompt': 'Describe the ophthalmological findings in this fundus image.',
 'answer': 'Cup-to-disc ratio 0.5, optic disc with well-defined margins and good color, central vessel emergence, preserved neuroretinal rim',
 'reference': 'Cup-to-disc ratio 0.5, optic disc with well-defined margins and good color, central vessel emergence, preserved neuroretinal rim',
 'locs_data': {}}

## 3. Conversaciones multimodales para SFT

In [5]:
def to_description_messages(row):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": Image.open(row["image"]).convert("RGB")},
                    {"type": "text", "text": row["prompt"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ]
    }

train_dataset = [to_description_messages(row) for row in train_rows]
print("training examples:", len(train_dataset))
train_dataset[0]


training examples: 1


{'messages': [{'role': 'user',
   'content': [{'type': 'image',
     'image': <PIL.Image.Image image mode=RGB size=3280x2480>},
    {'type': 'text',
     'text': 'Describe the ophthalmological findings in this fundus image.'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': 'Cup-to-disc ratio 0.5, optic disc with well-defined margins and good color, central vessel emergence, preserved neuroretinal rim'}]}]}

## 4. Cargar modelo completo + LoRA

In [6]:
major, minor = torch.cuda.get_device_capability()
TRAIN_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("GPU capability:", (major, minor), "TRAIN_DTYPE:", TRAIN_DTYPE)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=TRAIN_DTYPE,
    device_map="auto",
    token=HF_TOKEN,
)
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception as exc:
    print("gradient checkpointing no disponible:", exc)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)
print(type(model))


GPU capability: (8, 6) TRAIN_DTYPE: torch.bfloat16


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

<class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForConditionalGeneration'>


## 5. Collate function

In [7]:
def extract_image(messages):
    for msg in messages:
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]
        for element in content:
            if isinstance(element, dict) and element.get("type") == "image":
                return element["image"].convert("RGB")
    raise ValueError("No image found")

def collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        messages = example["messages"]
        texts.append(processor.apply_chat_template(
            messages,
            add_generation_prompt=False,
            tokenize=False,
        ).strip())
        images.append(extract_image(messages))

    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    for token_name in ("boi_token_id", "image_token_id", "eoi_token_id"):
        token_id = getattr(processor.tokenizer, token_name, None)
        if token_id is not None:
            labels[labels == token_id] = -100

    batch["labels"] = labels
    return batch

## 6. Helpers de generación y BERTScore

In [8]:
def first_model_device(model_obj):
    if hasattr(model_obj, "hf_device_map"):
        for device in model_obj.hf_device_map.values():
            if device not in {"cpu", "disk", "meta"}:
                return device
    if hasattr(model_obj, "device"):
        return model_obj.device
    return next(model_obj.parameters()).device

def generate_with_model(model_obj, processor_obj, prompt, image_path, max_new_tokens=384):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": Image.open(image_path).convert("RGB")},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor_obj.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    input_len = inputs["input_ids"].shape[-1]
    device = first_model_device(model_obj)
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                inputs[key] = value.to(device=device, dtype=TRAIN_DTYPE)
            else:
                inputs[key] = value.to(device=device)
    with torch.inference_mode():
        output = model_obj.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][input_len:]
    return processor_obj.decode(generated, skip_special_tokens=True).strip()

def evaluate_descriptions(outputs):
    candidates = [x["generated"] for x in outputs]
    references = [x["reference"] for x in outputs]
    P, R, F1 = bertscore(candidates, references, lang="en", rescale_with_baseline=True, verbose=True)
    for item, p, r, f in zip(outputs, P, R, F1):
        item["bertscore_precision"] = float(p)
        item["bertscore_recall"] = float(r)
        item["bertscore_f1"] = float(f)
    return {
        "count": len(outputs),
        "bertscore_precision_mean": float(P.mean()),
        "bertscore_recall_mean": float(R.mean()),
        "bertscore_f1_mean": float(F1.mean()),
    }, outputs

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

## 7. Baseline antes de entrenar

In [9]:
base_outputs = []
for idx, row in enumerate(test_rows, start=1):
    print(f"BASE [{idx}/{len(test_rows)}]", row["image"])
    generated = generate_with_model(model, processor, DESCRIPTION_PROMPT, row["image"], max_new_tokens=384)
    base_outputs.append({
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "reference": row["reference"],
        "generated": generated,
    })
    print(generated[:300])

base_summary, base_items = evaluate_descriptions(base_outputs)
base_summary

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


BASE [1/1] dataset\1217\1217_right.jpg


Based on the fundus image, here are the key ophthalmological findings:

*   **Retinal Hemorrhages:** There are several areas of retinal hemorrhages, particularly in the macula and periphery. These appear as red or dark red spots.
*   **Cotton Wool Spots:** There are some cotton wool spots, which are


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

...

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.25 seconds, 3.98 sentences/sec


C:\Users\Luco1421\Desktop\Pendientes\utils_medgemma\.venv\Lib\site-packages\bert_score\score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  baselines = torch.from_numpy(


{'count': 1,
 'bertscore_precision_mean': -0.19437694549560547,
 'bertscore_recall_mean': 0.028537774458527565,
 'bertscore_f1_mean': -0.08371104300022125}

## 8. Entrenar LoRA descriptivo

In [10]:
training_args = SFTConfig(
    output_dir=LORA_OUTPUT_DIR,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=2e-4,
    bf16=(TRAIN_DTYPE == torch.bfloat16),
    fp16=(TRAIN_DTYPE == torch.float16),
    logging_steps=1 if RUN_MODE == "smoke" else 5,
    save_steps=MAX_STEPS,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

trainer.train()
trainer.save_model(LORA_OUTPUT_DIR)
processor.save_pretrained(LORA_OUTPUT_DIR)
model = trainer.model
model.eval()
print("Adapter guardado en:", LORA_OUTPUT_DIR)


C:\Users\Luco1421\AppData\Local\Temp\ipykernel_15952\3784281203.py:1: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
1,5.275833


Adapter guardado en: checkpoints/official_medgemma_lora_description


## 9. Evaluar LoRA en test

In [11]:
lora_outputs = []
for idx, row in enumerate(test_rows, start=1):
    print(f"LORA [{idx}/{len(test_rows)}]", row["image"])
    generated = generate_with_model(model, processor, DESCRIPTION_PROMPT, row["image"], max_new_tokens=384)
    lora_outputs.append({
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "reference": row["reference"],
        "generated": generated,
    })
    print(generated[:300])

lora_summary, lora_items = evaluate_descriptions(lora_outputs)
comparison = {
    "base": base_summary,
    "lora": lora_summary,
    "delta_f1": lora_summary["bertscore_f1_mean"] - base_summary["bertscore_f1_mean"],
}
comparison


LORA [1/1] dataset\1217\1217_right.jpg


Based on the fundus image, here are the key ophthalmological findings:

*   **Retinal Hemorrhages:** There are several areas of retinal hemorrhages, which appear as red or blood-colored spots within the retina. These are indicative of bleeding within the retinal vessels.
*   **Cotton Wool Spots:** T


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.20 seconds, 4.96 sentences/sec


{'base': {'count': 1,
  'bertscore_precision_mean': -0.19437694549560547,
  'bertscore_recall_mean': 0.028537774458527565,
  'bertscore_f1_mean': -0.08371104300022125},
 'lora': {'count': 1,
  'bertscore_precision_mean': -0.1867133229970932,
  'bertscore_recall_mean': 0.032198112457990646,
  'bertscore_f1_mean': -0.07796403765678406},
 'delta_f1': 0.005747005343437195}

In [12]:
save_json("results/official_description_base_vs_lora_bertscore.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter_dir": LORA_OUTPUT_DIR,
    "run_mode": RUN_MODE,
    "split": "test",
    "train_examples": len(train_dataset),
    "test_examples": len(test_rows),
    "max_steps": MAX_STEPS,
    "comparison": comparison,
    "base_results": base_items,
    "lora_results": lora_items,
})
print(json.dumps(comparison, indent=2))


{
  "base": {
    "count": 1,
    "bertscore_precision_mean": -0.19437694549560547,
    "bertscore_recall_mean": 0.028537774458527565,
    "bertscore_f1_mean": -0.08371104300022125
  },
  "lora": {
    "count": 1,
    "bertscore_precision_mean": -0.1867133229970932,
    "bertscore_recall_mean": 0.032198112457990646,
    "bertscore_f1_mean": -0.07796403765678406
  },
  "delta_f1": 0.005747005343437195
}
